# Milestone 2: Evaluation plan

This notebook contains every number of the Milestone 2 report.

In [1]:
import pandas as pd 

df = pd.read_csv("../data/Pasion et al dataset.csv")

# Convert US units to metric, same as in notebook 01. 
# Altitude is already in metres in the file, so only wind, visibility and cloud ceiling are converted.
df["Wind.Speed"] = (df["Wind.Speed"] * 1.609).round(1)        # mph -> km/h
df["Visibility"] = (df["Visibility"] * 1.609).round(1)        # miles -> km
df["Cloud.Ceiling"] = (df["Cloud.Ceiling"] * 0.03048).round(1)  # 100 ft -> km

print(df.shape)

(21045, 17)


In [2]:
# Data: target, cleaning and copies (as in Milestone 1)
cutoff = df["PolyPwr"].median()
df["good_hour"] = (df["PolyPwr"] >= cutoff).astype(int)

df = df[df['Humidity']!= 0]

inputs = ["Latitude", "Longitude", "Altitude", "Month", "Hour",
          "Humidity", "AmbientTemp", "Wind.Speed", "Visibility",
          "Pressure", "Cloud.Ceiling"]

print(cutoff, len(df), df.duplicated(subset=inputs).sum())
print(df["good_hour"].value_counts())

13.7987 20993 0
good_hour
1    10506
0    10487
Name: count, dtype: int64


In [3]:
#Field 2: Split
from sklearn.model_selection import StratifiedGroupKFold

X, y, groups = df[inputs], df["good_hour"], df["Location"]

# 5 piles of whole 12 sites, each with a share of good hours close to the total amount;
# the first pile (about 20 percent) is the test part
cut = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(cut.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]

print("Test sites:", sorted(groups.iloc[test_idx].unique()))
print("Train rows: ", len(X_train), ", test rows: ",len(X_test))
print("Test share:", round(len(X_test) / len(X), 3))
print("Share of good hours,  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))
print(df.iloc[test_idx].groupby("Location")["Month"].nunique())

print(df.iloc[test_idx].groupby("Location")["good_hour"].mean().round(3))


Test sites: ['Grissom', 'Travis']
Train rows:  16760 , test rows:  4233
Test share: 0.202
Share of good hours,  train: 0.512  test: 0.456
Location
Grissom    12
Travis     12
Name: Month, dtype: int64
Location
Grissom    0.293
Travis     0.544
Name: good_hour, dtype: float64


In [4]:
# Field 3: Baseline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix

# 5 folds inside the training part, for 10 whole sites
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

lazy = DummyClassifier()   
s = cross_val_score(lazy, X_train, y_train, groups=g_train, cv=cv)
print("accuracy per fold:", s.round(3), " mean:", s.mean().round(3), " spread:", s.std().round(3))

pred = cross_val_predict(lazy, X_train, y_train, groups=g_train, cv=cv)
cm = confusion_matrix(y_train, pred)
print(cm)
print("weak hours called good:", round(cm[0, 1] / cm[0].sum(), 3))

accuracy per fold: [0.401 0.474 0.42  0.449 0.495]  mean: 0.448  spread: 0.034
[[3251 4933]
 [4265 4311]]
weak hours called good: 0.603


In [5]:
# per round: which sites are hidden, and the good share of the training part and the hidden part
for train_rows, hidden_rows in cv.split(X_train, y_train, g_train):
    print(sorted(g_train.iloc[hidden_rows].unique()),
          " train good share:", round(y_train.iloc[train_rows].mean(), 3),
          " hidden good share:", round(y_train.iloc[hidden_rows].mean(), 3))

['JDMT', 'MNANG']  train good share: 0.496  hidden good share: 0.599
['Kahului', 'Malmstrom', 'Offutt']  train good share: 0.521  hidden good share: 0.474
['Peterson']  train good share: 0.528  hidden good share: 0.42
['Hill Weber', 'USAFA']  train good share: 0.495  hidden good share: 0.551
['Camp Murray', 'March AFB']  train good share: 0.516  hidden good share: 0.495


In [6]:
#Field 4: Cross Validation of k nearest neighbors
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=1)   
s = cross_val_score(knn, X_train, y_train, groups=g_train, cv=cv)
print("accuracy per fold:", s.round(3), " mean:", s.mean().round(3), " spread:", s.std().round(3))

pred = cross_val_predict(knn, X_train, y_train, groups=g_train, cv=cv)
cm = confusion_matrix(y_train, pred)
print(cm)
print("weak hours called good:", round(cm[0, 1] / cm[0].sum(), 3))

accuracy per fold: [0.668 0.686 0.672 0.591 0.587]  mean: 0.641  spread: 0.043
[[5451 2733]
 [3411 5165]]
weak hours called good: 0.334


## Field 1 and 5: a concrete example

In [7]:
# Q: a weak Peterson hour that k = 1 on raw inputs called good (held out in round 3)
pred_raw = cross_val_predict(knn, X_train, y_train, groups=g_train, cv=cv)
mask = (y_train == 0) & (g_train == "Peterson") & (pred_raw == 1)
print("weak Peterson hours called good:", mask.sum())
Q = X_train[mask].iloc[[0]]

# round 3: the model knows all training sites except Peterson
known = g_train != "Peterson"
knn3 = KNeighborsClassifier(n_neighbors=3).fit(X_train[known], y_train[known])
dist, nb = knn3.kneighbors(Q)       

table = pd.concat([Q, X_train[known].iloc[nb[0]]])
table["site"] = g_train.loc[table.index].to_numpy()
table["good_hour"] = y_train.loc[table.index].to_numpy()
table["distance"] = [0] + dist[0].round(1).tolist()
table.index = ["Q", "A", "B", "C"]
print(table.T)

weak Peterson hours called good: 523
                      Q         A         B         C
Latitude          38.82     38.95     38.95     38.95
Longitude       -104.71   -104.83   -104.83   -104.83
Altitude           1879      1947      1947      1947
Month                 6         7         6         9
Hour                 13        12        14        13
Humidity       32.70874  34.86938    31.427  36.67603
AmbientTemp    25.19836  30.38231  27.75383  31.65123
Wind.Speed         17.7      14.5      14.5      20.9
Visibility         16.1      16.1      16.1      16.1
Pressure          818.7     807.8     806.5     808.4
Cloud.Ceiling       1.3       1.3       1.3       0.9
site           Peterson     USAFA     USAFA     USAFA
good_hour             0         1         0         1
distance            0.0      69.2      69.2      69.3


In [8]:
# Here is distance from Q to A, B, C on two inputs only
two = ["Humidity", "AmbientTemp"]
print((((table[two].iloc[1:] - table[two].iloc[0]) ** 2).sum(axis=1) ** 0.5).round(1))

A    5.6
B    2.9
C    7.6
dtype: float64


In [9]:
# Field 5: Cause of the mistake and preparation
diff2 = (table[inputs].iloc[1:] - table[inputs].iloc[0]) ** 2
print(diff2.round(1).T)
print("share of Altitude:", (diff2["Altitude"] / diff2.sum(axis=1)).round(3).to_dict())

                    A       B       C
Latitude          0.0     0.0     0.0
Longitude         0.0     0.0     0.0
Altitude       4624.0  4624.0  4624.0
Month             1.0     0.0     9.0
Hour              1.0     1.0     0.0
Humidity          4.7     1.6    15.7
AmbientTemp      26.9     6.5    41.6
Wind.Speed       10.2    10.2    10.2
Visibility        0.0     0.0     0.0
Pressure        118.8   148.8   106.1
Cloud.Ceiling     0.0     0.0     0.2
share of Altitude: {'A': 0.966, 'B': 0.965, 'C': 0.962}


In [10]:
# Standardisation inside a Pipeline, compared fold by fold with Field 4
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([("scale", StandardScaler()),
                 ("knn", KNeighborsClassifier(n_neighbors=1))])

s_raw = cross_val_score(knn, X_train, y_train, groups=g_train, cv=cv)
s_std = cross_val_score(pipe, X_train, y_train, groups=g_train, cv=cv)
print("raw:          ", s_raw.round(3), " mean:", s_raw.mean().round(3), " spread:", s_raw.std().round(3))
print("standardised: ", s_std.round(3), " mean:", s_std.mean().round(3), " spread:", s_std.std().round(3))
print("difference per fold:", (s_std - s_raw).round(3))

pred_std = cross_val_predict(pipe, X_train, y_train, groups=g_train, cv=cv)
cm = confusion_matrix(y_train, pred_std)
print(cm)
print("weak hours called good:", round(cm[0, 1] / cm[0].sum(), 3))

raw:           [0.668 0.686 0.672 0.591 0.587]  mean: 0.641  spread: 0.043
standardised:  [0.661 0.71  0.665 0.744 0.728]  mean: 0.702  spread: 0.033
difference per fold: [-0.007  0.024 -0.007  0.153  0.141]
[[5467 2717]
 [2159 6417]]
weak hours called good: 0.332


In [11]:
# round 3 again, now with the pipeline: the scaler learns only from the known sites
pipe.fit(X_train[known], y_train[known])
print(pd.Series(pipe["scale"].scale_, index=inputs).round(1))

dist_s, nb_s = pipe["knn"].kneighbors(pipe["scale"].transform(Q), n_neighbors=3)
near = X_train[known].iloc[nb_s[0]]
print(pd.DataFrame({"site": g_train.loc[near.index].to_numpy(),
                    "good_hour": y_train.loc[near.index].to_numpy(),
                    "distance": dist_s[0].round(2)}, index=["1st", "2nd", "3rd"]))

Latitude           7.7
Longitude         17.6
Altitude         713.0
Month              3.0
Hour               1.7
Humidity          23.0
AmbientTemp       12.7
Wind.Speed         9.9
Visibility         2.1
Pressure          80.5
Cloud.Ceiling      9.3
dtype: float64
      site  good_hour  distance
1st  USAFA          0      0.44
2nd  USAFA          0      0.54
3rd  USAFA          0      0.56


In [12]:
# Field 6: Tuning k
from sklearn.model_selection import GridSearchCV

# we try every k with the same pipeline and the same folds
grid = {"knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]}
search = GridSearchCV(pipe, grid, cv=cv).fit(X_train, y_train, groups=g_train)

# for every k: accuracy and spread from the grid search, plus both kinds of answers from cross_val_predict
rows = []
for k, mean, spread in zip(grid["knn__n_neighbors"],
                           search.cv_results_["mean_test_score"],
                           search.cv_results_["std_test_score"]):
    pipe_k = Pipeline([("scale", StandardScaler()),
                       ("knn", KNeighborsClassifier(n_neighbors=k))])
    pred_k = cross_val_predict(pipe_k, X_train, y_train, groups=g_train, cv=cv)
    cm = confusion_matrix(y_train, pred_k)
    rows.append([k, mean, spread, cm[0, 1] / cm[0].sum(), cm[1, 1] / cm[1].sum()])

table6 = pd.DataFrame(rows, columns=["k", "accuracy", "spread", "weak called good", "good recognised"])
print(table6.round(3).to_string(index=False))
print("best by accuracy:", search.best_params_, round(search.best_score_, 3))

 k  accuracy  spread  weak called good  good recognised
 1     0.702   0.033             0.332            0.748
 3     0.715   0.032             0.324            0.767
 5     0.729   0.029             0.316            0.785
 7     0.735   0.025             0.314            0.793
 9     0.737   0.024             0.313            0.797
11     0.738   0.023             0.313            0.798
15     0.740   0.022             0.314            0.802
21     0.740   0.023             0.320            0.808
31     0.744   0.022             0.321            0.815
51     0.745   0.025             0.320            0.817
best by accuracy: {'knn__n_neighbors': 51} 0.745
